<a href="https://colab.research.google.com/github/kansweldone/SRMMarks/blob/main/rag_lawbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q pymupdf
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DATA = Path('/content/drive/MyDrive/LawBot')
P3 = DATA / 'phase3'
ACTS = P3 / 'bare_acts'
ACTS.mkdir(parents=True, exist_ok=True)

%cd {ACTS}
from google.colab import files
uploaded = files.upload()      # click "Choose Files" and select all 3 PDFs
%cd /content

print(sorted(p.name for p in ACTS.iterdir()))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 62.4 MB/s eta 0:00:00
Mounted at /content/drive
/content/drive/MyDrive/LawBot/phase3/bare_acts


Saving constitution.pdf to constitution (1).pdf
Saving crpc.pdf to crpc (1).pdf
Saving IPC.pdf to IPC (1).pdf
/content
['IPC (1).pdf', 'IPC.pdf', 'constitution (1).pdf', 'constitution.pdf', 'crpc (1).pdf', 'crpc.pdf']


In [3]:
import fitz  # PyMuPDF

# Create the bare_acts_txt folder if it doesn't exist
TXT = P3 / 'bare_acts_txt'
TXT.mkdir(parents=True, exist_ok=True)

# Map the standard names to the actual uploaded file names
pdf_files = {
    'IPC': ACTS / 'IPC.pdf',
    'CrPC': ACTS / 'crpc.pdf',
    'Constitution': ACTS / 'constitution.pdf'
}

# Extract text from each PDF and save it as a text file
for act, pdf_path in pdf_files.items():
    if pdf_path.exists():
        print(f"Extracting text from {pdf_path.name}...")
        doc = fitz.open(pdf_path)
        text_content = []
        for page in doc:
            text_content.append(page.get_text())

        output_txt_path = TXT / f"{act}.txt"
        output_txt_path.write_text("\n".join(text_content), encoding='utf-8')
        print(f"Saved: {output_txt_path}")
    else:
        print(f"Warning: PDF not found at {pdf_path}")

Extracting text from IPC.pdf...
Saved: /content/drive/MyDrive/LawBot/phase3/bare_acts_txt/IPC.txt
Extracting text from crpc.pdf...
Saved: /content/drive/MyDrive/LawBot/phase3/bare_acts_txt/CrPC.txt
Extracting text from constitution.pdf...
Saved: /content/drive/MyDrive/LawBot/phase3/bare_acts_txt/Constitution.txt


Step2 : read every page and check the text quality

In [4]:
import re, json
from pathlib import Path

# Ensure paths are correctly defined
DATA = Path('/content/drive/MyDrive/LawBot')
P3 = DATA / 'phase3'
TXT = P3 / 'bare_acts_txt'

# Read the extracted text files
raw_text = {act: (TXT / f'{act}.txt').read_text(encoding='utf-8') for act in ['IPC', 'CrPC', 'Constitution']}

# amendment footnotes, e.g. "1. Subs. by Act 26 of 1955, s. 2" - not part of the law text
FOOT = re.compile(r'^\d+\s*\.?\s*(Subs\.|Ins\.|Omitted|Rep\.|The words|Added|Renumbered|Cl\.|Sub-s|See )')

def clean_text(t):
    lines = []
    for ln in t.split('\n'):
        s = ln.strip()
        if not s or re.fullmatch(r'\d{1,4}', s):   # empty lines and bare page numbers
            continue
        if FOOT.match(s):
            continue
        lines.append(s)
    t = '\n'.join(lines)
    t = re.sub(r'(\w)-\n(\w)', r'\1\2', t)            # re-join words split across lines
    t = re.sub(r'\d{1,2}\[([^\[\]]*)\]', r'\1', t)    # 1[amended text] -> amended text
    t = re.sub(r'(\*\s*){3,}', ' ', t)                # "* * *" omission markers
    return t
text = {act: clean_text(t) for act, t in raw_text.items()}
for act, t in text.items():
    print(f'{act:13s} raw {len(raw_text[act]):9,d} chars -> clean {len(t):9,d} chars')

IPC           raw   502,482 chars -> clean   468,034 chars
CrPC          raw   933,383 chars -> clean   885,399 chars
Constitution  raw   877,548 chars -> clean   798,879 chars


Step 5: split into sections and check against known sections

In [5]:
HEAD = re.compile(r'^(\d{1,3}[A-Z]{0,3})\.\s+([\[A-Z“"‘\'][^—–\n]{2,200}?(?:\n[^—–\n]{1,120}?)?)[.:\]’”"]?\s*[—–]', re.M)
SCHED = re.compile(r'^(?:\d{0,2}\[)?(?:THE\s+)?(FIRST|SECOND|THIRD|FOURTH|FIFTH|SIXTH|SEVENTH|EIGHTH|NINTH|TENTH|ELEVENTH|TWELFTH)\s+SCHEDULE', re.M)
LABEL = {'IPC': 'Section', 'CrPC': 'Section', 'Constitution': 'Article'}
LAST = {'IPC': '511', 'CrPC': '484', 'Constitution': '395'}   # last numbered section/article of each Act

def split_act(act, t):
    ms = list(HEAD.finditer(t))
    # 1. Schedules start at the first SCHEDULE heading after the LAST "484." / "395." heading
    cut = len(t)
    last = [m for m in ms if m.group(1) == LAST[act]]
    if last:
        sm = SCHED.search(t, last[-1].end())
        if sm:
            cut = sm.start()
    print(f'{act}: {len(last)} match(es) for "{LAST[act]}.", schedules start at {cut / len(t):.0%} of the text')
    main = [m for m in ms if m.start() < cut]

    # 2. sections/articles from the main text; duplicates (e.g. contents pages) -> keep the longest
    secs = {}
    for i, m in enumerate(main):
        end = main[i + 1].start() if i + 1 < len(main) else cut
        s = {'act': act, 'num': m.group(1), 'ref': f"{LABEL[act]} {m.group(1)}",
             'title': ' '.join(m.group(2).split()).strip('[]“”"‘’\''), 'body': t[m.start():end].strip()}
        if s['num'] not in secs or len(s['body']) > len(secs[s['num']]['body']):
            secs[s['num']] = s
    out = list(secs.values())

    # 3. schedules as separate pieces
    sch = list(SCHED.finditer(t, cut))
    for i, m in enumerate(sch):
        end = sch[i + 1].start() if i + 1 < len(sch) else len(t)
        name = f"{m.group(1).title()} Schedule"
        out.append({'act': act, 'num': name, 'ref': name, 'title': name, 'body': t[m.start():end].strip()})
    return out

sections = [s for act in text for s in split_act(act, text[act])]

CHECK = {'IPC': ['10', '302', '378', '420', '498A'], 'CrPC': ['154', '357', '438'], 'Constitution': ['14', '21', '32', '368']}
for act in CHECK:
    found = {s['num']: s for s in sections if s['act'] == act}
    nums = [n for n in found if n[0].isdigit()]
    scheds = [n for n in found if not n[0].isdigit()]
    plain = {int(re.match(r'\d+', n).group()) for n in nums}
    missing = [i for i in range(1, int(LAST[act]) + 1) if i not in plain]
    print(f'\n{act}: {len(nums)} {LABEL[act].lower()}s, schedules: {scheds}')
    print(f'  missing numbers (1-{LAST[act]}): {len(missing)} -> {missing[:30]}{" ..." if len(missing) > 30 else ""}')
    for n in CHECK[act]:
        print(f"  {LABEL[act]} {n:4s}:", found[n]['title'][:70] if n in found else '*** MISSING ***')

IPC: 1 match(es) for "511.", schedules start at 100% of the text
CrPC: 2 match(es) for "484.", schedules start at 80% of the text
Constitution: 1 match(es) for "395.", schedules start at 73% of the text

IPC: 531 sections, schedules: []
  missing numbers (1-511): 28 -> [4, 5, 14, 15, 16, 17, 40, 56, 58, 60, 61, 62, 75, 161, 162, 163, 164, 165, 215, 226, 292, 293, 375, 478, 480, 490, 492, 505]
  Section 10  : Man”. “Woman
  Section 302 : Punishment for murder
  Section 378 : Theft
  Section 420 : Cheating and dishonestly inducing delivery of property
  Section 498A: Husband or relative of husband of a woman subjecting her to cruelty

CrPC: 531 sections, schedules: ['First Schedule', 'Second Schedule']
  missing numbers (1-484): 1 -> [24]
  Section 154 : Information in cognizable cases
  Section 357 : Order to pay compensation
  Section 438 : Direction for grant of bail to person apprehending arrest

Constitution: 455 articles, schedules: ['First Schedule', 'Second Schedule', 'Third Sche

step 6: cut into token-sized chunks and save

In [6]:
!pip install -q langchain-text-splitters sentence-transformers
from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np

EMB_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
emb_tok = AutoTokenizer.from_pretrained(EMB_MODEL)
splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(emb_tok, chunk_size=450, chunk_overlap=80)

chunks = []
for s in sections:
    ref = s['ref']                                   # 'Section 302', 'Article 21' or 'Seventh Schedule'
    header = f"[{s['act']} | {ref} | {s['title']}]"
    for k, part in enumerate(splitter.split_text(s['body'])):
        chunks.append({'id': f"{s['act']}-{s['num']}-{k}", 'act': s['act'], 'ref': ref,
                       'title': s['title'], 'part': k, 'text': f'{header}\n{part}'})

ntok = [len(emb_tok(c['text']).input_ids) for c in chunks]
print(f'{len(sections)} sections -> {len(chunks)} chunks')
print(f'tokens per chunk: median {int(np.median(ntok))}, 95th pct {int(np.percentile(ntok, 95))}, max {max(ntok)}')
print({a: sum(c['act'] == a for c in chunks) for a in LABEL})

with open(P3 / 'chunks.jsonl', 'w', encoding='utf-8') as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + '\n')

print('\nSchedule labels:', sorted({c['ref'] for c in chunks if 'Schedule' in c['ref']}))
print('\nSample chunk:\n', next(c for c in chunks if c['id'] == 'IPC-302-0')['text'][:300])

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1534 sections -> 2110 chunks
tokens per chunk: median 226, 95th pct 464, max 490
{'IPC': 600, 'CrPC': 776, 'Constitution': 734}

Schedule labels: ['Eighth Schedule', 'Eleventh Schedule', 'Fifth Schedule', 'First Schedule', 'Fourth Schedule', 'Ninth Schedule', 'Second Schedule', 'Seventh Schedule', 'Sixth Schedule', 'Tenth Schedule', 'Third Schedule', 'Twelfth Schedule']

Sample chunk:
 [IPC | Section 302 | Punishment for murder]
302. Punishment for murder.—Whoever commits murder shall be punished with death or
imprisonment for life,  and shall also be liable to fine.


Step 3: Embed the chunks and build the FAISS index

In [7]:
!pip install -q faiss-cpu langchain-community langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


Cell 8: embedding model, wrapped for LangChain



Cell 21: embedding model, wrapped for LangChain

In [8]:
from sentence_transformers import SentenceTransformer
from langchain_core.embeddings import Embeddings

class MiniLMEmbeddings(Embeddings):
    def __init__(self, name=EMB_MODEL):
        self.model = SentenceTransformer(name, device='cuda')
        self.model.max_seq_length = 512          # default is 256 -> would ignore the end of longer chunks
    def embed_documents(self, texts):
        return self.model.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True).tolist()
    def embed_query(self, text):
        return self.model.encode(text, normalize_embeddings=True).tolist()

emb = MiniLMEmbeddings()
print('max_seq_length:', emb.model.max_seq_length, '| dim:', emb.model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

max_seq_length: 512 | dim: 384


/tmp/ipykernel_1330/3406144034.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print('max_seq_length:', emb.model.max_seq_length, '| dim:', emb.model.get_sentence_embedding_dimension())


Cell 22: build the FAISS index and save it

In [9]:
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

vs = FAISS.from_texts(
    texts=[c['text'] for c in chunks],
    embedding=emb,
    metadatas=[{k: c[k] for k in ('id', 'act', 'ref', 'title', 'part')} for c in chunks],
    distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT,   # normalized vectors -> inner product = cosine similarity
)
vs.save_local(str(P3 / 'faiss_index'))
print('vectors in index:', vs.index.ntotal)

/tmp/ipykernel_1330/217844208.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Batches:   0%|          | 0/33 [00:00<?, ?it/s]

vectors in index: 2110


Cell 23: test retrieval with known questions



In [10]:
TESTS = [
    ('What is the punishment for murder?',                                   'IPC',          'Section 302'),
    ('What is theft?',                                                       'IPC',          'Section 378'),
    ('When can a court order the accused to pay compensation to the victim?', 'CrPC',         'Section 357'),
    ('How can a person apprehending arrest get anticipatory bail?',          'CrPC',         'Section 438'),
    ('Which article protects life and personal liberty?',                    'Constitution', 'Article 21'),
    ('Can Parliament amend the Constitution?',                               'Constitution', 'Article 368'),
]
hits = 0
for q, act, ref in TESTS:
    res = vs.similarity_search_with_score(q, k=3)
    got = [(d.metadata['act'], d.metadata['ref']) for d, _ in res]
    ok = (act, ref) in got
    hits += ok
    print(f"{'✓' if ok else '✗'} {q}\n   expected {act} {ref} | top-3: " +
          ', '.join(f"{d.metadata['act']} {d.metadata['ref']} ({s:.2f})" for d, s in res))
print(f'\nHit@3: {hits}/{len(TESTS)}')

✓ What is the punishment for murder?
   expected IPC Section 302 | top-3: IPC Section 302 (0.73), IPC Section 303 (0.70), IPC Section 304 (0.64)
✗ What is theft?
   expected IPC Section 378 | top-3: IPC Section 356 (0.61), IPC Section 381 (0.61), IPC Section 413 (0.60)
✗ When can a court order the accused to pay compensation to the victim?
   expected CrPC Section 357 | top-3: CrPC Section 237 (0.69), CrPC Section 250 (0.66), CrPC Section 250 (0.58)
✓ How can a person apprehending arrest get anticipatory bail?
   expected CrPC Section 438 | top-3: CrPC Section 438 (0.67), CrPC Section 89 (0.64), CrPC Section 50 (0.63)
✓ Which article protects life and personal liberty?
   expected Constitution Article 21 | top-3: Constitution Article 21 (0.76), Constitution Article 38 (0.45), Constitution Article 42 (0.45)
✓ Can Parliament amend the Constitution?
   expected Constitution Article 368 | top-3: Constitution Article 247 (0.64), Constitution Article 368 (0.59), Constitution Article 368 (0.5

Step 3b: Hybrid search and re-ranking

Cell 24: BM25 + cross-encoder + a single retrieve() function

In [11]:
!pip install -q rank_bm25
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

tokenize = lambda s: re.findall(r'[a-z0-9]+', s.lower())
bm25 = BM25Okapi([tokenize(c['text']) for c in chunks])
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device='cuda', max_length=512)
E = vs.index.reconstruct_n(0, vs.index.ntotal)        # all chunk vectors, same order as `chunks`

def retrieve(query, k=3, mode='rerank', n_cand=30):
    qv = np.array(emb.embed_query(query))
    dense = E @ qv
    d_rank = np.argsort(-dense)[:n_cand]
    if mode == 'dense':
        return [(int(i), float(dense[i])) for i in d_rank[:k]]
    b_rank = np.argsort(-bm25.get_scores(tokenize(query)))[:n_cand]
    rrf = {}                                            # Reciprocal Rank Fusion of both lists
    for ranking in (d_rank, b_rank):
        for r, i in enumerate(ranking):
            rrf[int(i)] = rrf.get(int(i), 0) + 1 / (60 + r)
    cand = sorted(rrf, key=rrf.get, reverse=True)[:n_cand]
    if mode == 'hybrid':
        return [(i, rrf[i]) for i in cand[:k]]
    ce = reranker.predict([(query, chunks[i]['text']) for i in cand])
    return [(cand[j], float(ce[j])) for j in np.argsort(-ce)[:k]]

for mode in ['dense', 'hybrid', 'rerank']:
    hits = 0
    for q, act, ref in TESTS:
        got = [(chunks[i]['act'], chunks[i]['ref']) for i, _ in retrieve(q, mode=mode)]
        hits += (act, ref) in got
    print(f'{mode:7s} Hit@3 on the 6 test questions: {hits}/6')

print('\nrerank, top-3 for the two earlier misses:')
for q, act, ref in [TESTS[1], TESTS[2]]:
    print(' ', q, '->', [chunks[i]['ref'] for i, _ in retrieve(q)])

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

dense   Hit@3 on the 6 test questions: 4/6
hybrid  Hit@3 on the 6 test questions: 5/6
rerank  Hit@3 on the 6 test questions: 5/6

rerank, top-3 for the two earlier misses:
  What is theft? -> ['Section 380', 'Section 381', 'Section 410']
  When can a court order the accused to pay compensation to the victim? -> ['Section 357', 'Section 357', 'Section 357']


Cell 25: evaluate on my validation questions



In [12]:
import re, json
import pandas as pd
from tqdm.auto import tqdm

SEC_RE = re.compile(r'(?<![-\w])(Section|Sec\.|Article|Art\.)\s+([0-9]+[A-Z]{0,2})\b', re.I)
def get_refs(text):
    return {('Article' if k.lower().startswith('art') else 'Section') + ' ' + n.upper() for k, n in SEC_RE.findall(text)}

val = [json.loads(l) for l in open(DATA / 'lawbot_val.jsonl', encoding='utf-8')]
evalq = []
for r in val:
    gold = get_refs(r['output']) - get_refs(r['instruction'])
    want = 'Article' if r['source'] == 'Constitution' else 'Section'
    gold = {g for g in gold if g.startswith(want)}
    if gold:
        evalq.append((r['instruction'], r['source'], gold))
evalq = evalq[:300]
print('retrieval eval questions:', len(evalq))

results = {}
for mode in ['dense', 'hybrid', 'rerank']:
    h1 = h3 = h5 = 0
    for q, act, gold in tqdm(evalq, desc=mode):
        got = [(chunks[i]['act'], chunks[i]['ref']) for i, _ in retrieve(q, k=5, mode=mode)]
        hit_at = [any(a == act and rf in gold for a, rf in got[:n]) for n in (1, 3, 5)]
        h1, h3, h5 = h1 + hit_at[0], h3 + hit_at[1], h5 + hit_at[2]
    results[mode] = {'Hit@1': round(h1 / len(evalq), 3), 'Hit@3': round(h3 / len(evalq), 3), 'Hit@5': round(h5 / len(evalq), 3)}

res_df = pd.DataFrame(results).T
res_df.to_csv(P3 / 'retrieval_eval.csv')
display(res_df)

retrieval eval questions: 195


dense:   0%|          | 0/195 [00:00<?, ?it/s]

hybrid:   0%|          | 0/195 [00:00<?, ?it/s]

rerank:   0%|          | 0/195 [00:00<?, ?it/s]

,Hit@1,Hit@3,Hit@5
dense,0.313,0.400,0.431
hybrid,0.344,0.436,0.497
rerank,0.385,0.462,0.533


Cell 27: look at 10 failures



In [13]:
fails = []
for q, act, gold in evalq:
    got = [(chunks[i]['act'], chunks[i]['ref'], chunks[i]['title'][:40]) for i, _ in retrieve(q, k=5)]
    if not any(a == act and rf in gold for a, rf, _ in got):
        fails.append((q, act, gold, got[:3]))
print(f'{len(fails)} failures out of {len(evalq)}\n')
for q, act, gold, got in fails[:10]:
    print(f'Q: {q}\n   expected: {act} {sorted(gold)}\n   got: {[f"{a} {r} ({t})" for a, r, t in got]}\n')

91 failures out of 195

Q: When can an appeal lie to the Supreme Court from any judgment, decree or final order of a High Court in the territory of India?
   expected: Constitution ['Article 134A']
   got: ['Constitution Article 132 (Appellate jurisdiction of Supreme Court )', 'Constitution Article 133 (Appellate jurisdiction of Supreme Court )', 'Constitution Article 134 (Appellate jurisdiction of Supreme Court )']

Q: Under what circumstances can the Governor authorise expenditure for a particular service for the current financial year?
   expected: Constitution ['Article 204']
   got: ['Constitution Article 205 (Supplementary, additional or excess gran)', 'Constitution Article 115 (Supplementary, additional or excess gran)', 'Constitution Article 202 (Annual financial statement)']

Q: What was repealed by the Constitution (Forty-third Amendment) Act, 1977, s. 8?
   expected: Constitution ['Article 226']
   got: ['Constitution Article 225 (Jurisdiction of existing High Courts)', 'Con

In [14]:
print(sorted({c['ref'] for c in chunks if 'Schedule' in c['ref']}))

['Eighth Schedule', 'Eleventh Schedule', 'Fifth Schedule', 'First Schedule', 'Fourth Schedule', 'Ninth Schedule', 'Second Schedule', 'Seventh Schedule', 'Sixth Schedule', 'Tenth Schedule', 'Third Schedule', 'Twelfth Schedule']


Cell 30: load the Phase 2 LawBot (base Qwen plus your LoRA adapter)



In [15]:
!pip install -q peft bitsandbytes
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE = 'unsloth/Qwen2.5-1.5B-Instruct'             # same base model as Phase 2
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
llm_tok = AutoTokenizer.from_pretrained(BASE)
base_model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map='cuda')
llm = PeftModel.from_pretrained(base_model, str(DATA / 'lawbot_lora_adapter'))
llm.eval()
llm.generation_config.max_length = None             # silences the max_length warning from Phase 2
print('loaded | GPU memory used: %.1f GB' % (torch.cuda.memory_allocated() / 1e9))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 23.4 MB/s eta 0:00:00


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.36k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

loaded | GPU memory used: 1.5 GB


Cell 31: compare top scores for real and unrelated questions



In [16]:
in_scores = [retrieve(q, k=1)[0][1] for q, _, _ in tqdm(evalq[:100], desc='legal')]

OFF_TOPIC = [
    'What is the capital of France?', 'How do I bake sourdough bread?', 'Who won the 2011 Cricket World Cup?',
    'What is the GST rate on gold jewellery?', 'How do I file income tax returns online?',
    'What are the grounds for divorce under the Hindu Marriage Act?', 'How do I register a trademark in India?',
    'What is the minimum wage in Tamil Nadu?', 'Explain the theory of relativity.', 'How do I reset my bank PIN?',
]
off_scores = [retrieve(q, k=1)[0][1] for q in OFF_TOPIC]

print('legal questions   (top score): min %.2f | 10th pct %.2f | median %.2f' %
      (min(in_scores), np.percentile(in_scores, 10), np.median(in_scores)))
print('off-topic questions(top score): max %.2f | median %.2f' % (max(off_scores), np.median(off_scores)))
for q, s in sorted(zip(OFF_TOPIC, off_scores), key=lambda x: -x[1]):
    print(f'  {s:6.2f}  {q}')

legal:   0%|          | 0/100 [00:00<?, ?it/s]

legal questions   (top score): min -2.94 | 10th pct 3.82 | median 7.03
off-topic questions(top score): max 0.07 | median -9.16
    0.07  What are the grounds for divorce under the Hindu Marriage Act?
   -4.97  What is the minimum wage in Tamil Nadu?
   -5.38  How do I register a trademark in India?
   -8.64  How do I file income tax returns online?
   -9.08  Explain the theory of relativity.
   -9.24  What is the capital of France?
  -10.06  Who won the 2011 Cricket World Cup?
  -10.66  How do I reset my bank PIN?
  -11.01  How do I bake sourdough bread?
  -11.25  What is the GST rate on gold jewellery?


Step 4 (part 2): the LawBot answer pipeline
Cell 32: set the threshold and count how many legal questions it would decline



In [17]:
ABSTAIN_T = 1.0
in_arr = np.array(in_scores)
print(f'threshold {ABSTAIN_T}: legal questions declined = {(in_arr < ABSTAIN_T).mean():.1%}, '
      f'off-topic declined = {(np.array(off_scores) < ABSTAIN_T).mean():.0%}')

threshold 1.0: legal questions declined = 4.0%, off-topic declined = 100%


Cell 33: the pipeline (rewrite, then retrieve, then answer, then check citations)


In [18]:
def chat(msgs, max_new_tokens=256):
    prompt = llm_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = llm_tok(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=llm_tok.eos_token_id)
    return llm_tok.decode(out[0, enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()

# 1. history-aware rewrite: turn a follow-up into a standalone question
REWRITE_SYS = ("Rewrite the user's latest question as one standalone question about Indian law. "
               "Replace words like 'it', 'they', 'that' or 'this offence' with the exact thing they refer to in the conversation. "
               "Output only the rewritten question.\n\n"
               "Example:\n"
               "Conversation:\nUser: What is theft?\nLawBot: Theft is dishonestly taking movable property out of a person's possession without consent [IPC Section 378].\n"
               "Latest question: Is it a cognizable offence?\n"
               "Standalone question: Is theft a cognizable offence?")
def rewrite(question, history):
    if not history:
        return question
    convo = '\n'.join(f'{role}: {text[:300]}' for role, text in history[-4:])
    msgs = [{'role': 'system', 'content': REWRITE_SYS},
            {'role': 'user', 'content': f'Conversation:\n{convo}\n\nLatest question: {question}\n\nStandalone question:'}]
    with llm.disable_adapter():                     # general rewriting task -> plain base model
        return chat(msgs, 60).split('\n')[0].strip()

# 2. retrieve and group chunks into up to 3 distinct sections
def gather_sections(query, n_sections=3):
    res = retrieve(query, k=10)
    picked = {}
    for i, score in res:
        key = (chunks[i]['act'], chunks[i]['ref'])
        if key not in picked and len(picked) == n_sections:
            continue
        picked.setdefault(key, {'act': key[0], 'ref': key[1], 'title': chunks[i]['title'], 'score': score, 'parts': []})
        picked[key]['parts'].append(chunks[i])
    secs = []
    for p in picked.values():
        body = '\n'.join(c['text'].split('\n', 1)[1] for c in sorted(p['parts'], key=lambda c: c['part']))
        secs.append({**{k: p[k] for k in ('act', 'ref', 'title', 'score')}, 'text': body[:2500]})
    return (res[0][1] if res else -99), secs

# 3. grounded answer with citations
ANSWER_SYS = ("You are LawBot, an assistant for Indian law (IPC, CrPC and the Constitution of India). "
              "Answer the question using ONLY the legal text in the context. "
              "Cite every Section or Article you rely on in square brackets, e.g. [IPC Section 302] or [Constitution Article 21]. "
              "If the context does not answer the question, reply exactly: I could not find this in the provided legal text.")
NOT_FOUND = ("I could not find this in the IPC, CrPC or Constitution text available to me. "
             "Please refer to the relevant Act or consult a qualified lawyer.")
DISCLAIMER = 'LawBot is for educational use only and is not legal advice.'
CITE_RE = re.compile(r'\[(IPC|CrPC|Constitution)\s+((?:Section|Article)\s+[0-9A-Z]+|\w+\s+Schedule)\]')

def lawbot(question, history=None, use_adapter=False):
    history = history or []
    standalone = rewrite(question, history)
    top, secs = gather_sections(standalone)
    if history and top < ABSTAIN_T:                 # rewrite may have failed -> retry with the previous user question added
        last_q = next((t for role, t in reversed(history) if role == 'User'), '')
        top2, secs2 = gather_sections(f'{last_q} {standalone}')
        if top2 > top:
            top, secs = top2, secs2
    if top < ABSTAIN_T or not secs:
        return {'question': question, 'standalone': standalone, 'answer': NOT_FOUND, 'sources': [],
                'top_score': round(top, 2), 'abstained': True, 'unsupported_citations': []}
    ctx = '\n\n'.join(f"[{s['act']} {s['ref']} - {s['title']}]\n{s['text']}" for s in secs)
    msgs = [{'role': 'system', 'content': ANSWER_SYS},
            {'role': 'user', 'content': f'Context:\n{ctx}\n\nQuestion: {standalone}'}]
    if use_adapter:
        ans = chat(msgs)
    else:
        with llm.disable_adapter():
            ans = chat(msgs)
    given = {(s['act'], s['ref']) for s in secs}
    cited = set(CITE_RE.findall(ans))
    return {'question': question, 'standalone': standalone, 'answer': ans,
            'sources': [f"{s['act']} {s['ref']} ({s['score']:.1f})" for s in secs],
            'top_score': round(top, 2), 'abstained': 'could not find' in ans.lower(),
            'unsupported_citations': sorted(f'{a} {r}' for a, r in cited - given)}

def show(r):
    print(f"Q: {r['question']}")
    if r['standalone'] != r['question']:
        print(f"   rewritten -> {r['standalone']}")
    print(f"A: {r['answer']}\n   sources: {r['sources']} | top score {r['top_score']} | abstained: {r['abstained']}")
    if r['unsupported_citations']:
        print(f"   ⚠ cited but NOT in context: {r['unsupported_citations']}")
    print(f'   ({DISCLAIMER})\n')

Cell 34: try it on single questions, a conversation and off-topic questions

In [19]:
for q in ['What is the punishment for murder?',
          'What is the GST rate on gold jewellery?',
          'What are the grounds for divorce under the Hindu Marriage Act?']:
    show(lawbot(q))

print('=' * 30, 'multi-turn conversation', '=' * 30)
history = []
for q in ['What is anticipatory bail?', 'Which courts can grant it?', 'Can they impose conditions?']:
    r = lawbot(q, history)
    show(r)
    history += [('User', q), ('LawBot', r['answer'])]

Q: What is the punishment for murder?
A: The punishment for murder includes either death or imprisonment for life, as per [IPC Section 302]. Additionally, if someone commits murder while under sentence of imprisonment for life, they will face the same punishment as specified in [IPC Section 303], which involves a mandatory death sentence.
   sources: ['IPC Section 302 (9.0)', 'IPC Section 303 (7.4)', 'IPC Section 396 (6.1)'] | top score 9.01 | abstained: False
   (LawBot is for educational use only and is not legal advice.)

Q: What is the GST rate on gold jewellery?
A: I could not find this in the IPC, CrPC or Constitution text available to me. Please refer to the relevant Act or consult a qualified lawyer.
   sources: [] | top score -11.25 | abstained: True
   (LawBot is for educational use only and is not legal advice.)

Q: What are the grounds for divorce under the Hindu Marriage Act?
A: I could not find this in the IPC, CrPC or Constitution text available to me. Please refer to th

cell 19 code

In [20]:
for q in ['What is the punishment for murder?']:
    show(lawbot(q, use_adapter=False))

print('=' * 30, 'multi-turn conversation (base model)', '=' * 30)
history = []
for q in ['What is anticipatory bail?', 'Which courts can grant it?', 'Can they impose conditions?']:
    r = lawbot(q, history, use_adapter=False)
    show(r)
    history += [('User', q), ('LawBot', r['answer'])]

Q: What is the punishment for murder?
A: The punishment for murder includes either death or imprisonment for life, as per [IPC Section 302]. Additionally, if someone commits murder while under sentence of imprisonment for life, they will face the same punishment as specified in [IPC Section 303], which involves a mandatory death sentence.
   sources: ['IPC Section 302 (9.0)', 'IPC Section 303 (7.4)', 'IPC Section 396 (6.1)'] | top score 9.01 | abstained: False
   (LawBot is for educational use only and is not legal advice.)

============================== multi-turn conversation (base model) ==============================
Q: What is anticipatory bail?
A: Anticipatory bail refers to a situation where a court issues an interim order allowing someone to be released on bail pending further proceedings, specifically when they have reasonable grounds to fear arrest due to an impending criminal charge. This allows them to leave custody while their case is being considered.
   sources: ['CrPC 

Cell 19: same questions, base Qwen + RAG



In [21]:
for q in ['What is the punishment for murder?']:
    show(lawbot(q, use_adapter=False))

print('=' * 30, 'multi-turn conversation (base model)', '=' * 30)
history = []
for q in ['What is anticipatory bail?', 'Which courts can grant it?', 'Can they impose conditions?']:
    r = lawbot(q, history, use_adapter=False)
    show(r)
    history += [('User', q), ('LawBot', r['answer'])]

Q: What is the punishment for murder?
A: The punishment for murder includes either death or imprisonment for life, as per [IPC Section 302]. Additionally, if someone commits murder while under sentence of imprisonment for life, they will face the same punishment as specified in [IPC Section 303], which involves a mandatory death sentence.
   sources: ['IPC Section 302 (9.0)', 'IPC Section 303 (7.4)', 'IPC Section 396 (6.1)'] | top score 9.01 | abstained: False
   (LawBot is for educational use only and is not legal advice.)

============================== multi-turn conversation (base model) ==============================
Q: What is anticipatory bail?
A: Anticipatory bail refers to a situation where a court issues an interim order allowing someone to be released on bail pending further proceedings, specifically when they have reasonable grounds to fear arrest due to an impending criminal charge. This allows them to leave custody while their case is being considered.
   sources: ['CrPC 

Cell 20: scoring setup



In [22]:
!pip install -q evaluate rouge_score bert_score
import evaluate
rouge = evaluate.load('rouge')
bertscore = evaluate.load('bertscore')

def score(preds, golds):
    rl = rouge.compute(predictions=preds, references=golds)['rougeL']
    bs = float(np.mean(bertscore.compute(predictions=preds, references=golds, lang='en')['f1']))
    pairs = [(get_refs(g), get_refs(p)) for g, p in zip(golds, preds) if get_refs(g)]
    cit = float(np.mean([len(g & p) / len(g) for g, p in pairs])) if pairs else float('nan')
    return {'rougeL': round(rl, 4), 'bertscore_f1': round(bs, 4),
            'citation_recall': round(cit, 4), 'n_with_citations': len(pairs)}

ev = pd.read_csv(DATA / 'eval_predictions_base_vs_lawbot.csv')     # the 200 Phase 2 questions + answers
print(len(ev), 'questions loaded')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.4 MB/s eta 0:00:00


200 questions loaded


Cell 21: run RAG with both models (about 20–30 minutes, so keep the tab open)


In [23]:
for name, use_ad in [('base_rag', False), ('lawbot_rag', True)]:
    outs = [lawbot(q, use_adapter=use_ad) for q in tqdm(ev.question, desc=name)]
    ev[f'{name}_pred'] = [o['answer'] for o in outs]
    ev[f'{name}_abstained'] = [o['abstained'] for o in outs]
    ev[f'{name}_has_cite'] = [bool(CITE_RE.search(o['answer'])) for o in outs]
    ev[f'{name}_unsupported'] = [len(o['unsupported_citations']) for o in outs]
    ev[f'{name}_sources'] = [' | '.join(o['sources']) for o in outs]
ev.to_csv(P3 / 'eval_rag_predictions.csv', index=False)
print('saved')

base_rag:   0%|          | 0/200 [00:00<?, ?it/s]

KeyboardInterrupt: 

Cell 22: the comparison table


In [25]:
rows = {}
for label, col in [('Base (closed-book)', 'base_pred'), ('LawBot FT (closed-book)', 'lawbot_pred'),
                   ('Base + RAG', 'base_rag_pred'), ('LawBot FT + RAG', 'lawbot_rag_pred')]:
    r = score(ev[col].fillna('').tolist(), ev.gold.tolist())
    key = col[:-5]
    if f'{key}_abstained' in ev:
        r['abstain_rate'] = round(ev[f'{key}_abstained'].mean(), 3)
        r['answers_with_citation'] = round(ev[f'{key}_has_cite'].mean(), 3)
        r['unsupported_citations'] = int(ev[f'{key}_unsupported'].sum())
    rows[label] = r
table = pd.DataFrame(rows).T
table.to_csv(P3 / 'phase3_eval_results.csv')
table

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


KeyError: 'base_rag_pred'

In [ ]:
bad = ev[ev.base_rag_unsupported > 0]
for _, r in bad.iterrows():
    print(f"Q: {r.question}\n   cited: {CITE_RE.findall(r.base_rag_pred)}\n   given: {r.base_rag_sources}\n")

print('=' * 20, 'declined questions (first 5)', '=' * 20)
for _, r in ev[ev.base_rag_abstained].head(5).iterrows():
    print(f'Q: {r.question}\n   gold: {r.gold}\n')

In [26]:
import pandas as pd
pd.read_csv(P3 / 'phase3_eval_results.csv', index_col=0)

,rougeL,bertscore_f1,citation_recall,n_with_citations,abstain_rate,answers_with_citation,unsupported_citations
Base (closed-book),0.1450,0.8494,0.3077,26.0,NaN,NaN,NaN
LawBot FT (closed-book),0.3173,0.8918,0.3077,26.0,NaN,NaN,NaN
Base + RAG,0.3389,0.8834,0.6538,26.0,0.07,0.175,6.0
LawBot FT + RAG,0.4618,0.9117,0.5000,26.0,0.07,0.000,0.0


Step 1: Build and launch the app
Cell 24: install


In [27]:
!pip install -q gradio
import gradio as gr
print(gr.__version__)

6.29.0


Cell 25: the app



In [28]:
def format_trace(r):
    lines = []
    if r['standalone'] != r['question']:
        lines.append(f"**Searched for:** {r['standalone']}")
    if r['sources']:
        lines.append('**Retrieved sources** (re-ranker score):')
        lines += [f'- {s}' for s in r['sources']]
    else:
        lines.append(f"**No relevant text found** (top score {r['top_score']} < threshold {ABSTAIN_T})")
    if r['unsupported_citations']:
        lines.append(f"⚠️ **Cited but not in the retrieved text:** {', '.join(r['unsupported_citations'])}. Verify before relying on it.")
    return '\n'.join(lines)

def respond(message, chat_history, lb_history, model_choice):
    if not message.strip():
        return '', chat_history, lb_history, gr.update()
    r = lawbot(message, lb_history, use_adapter=(model_choice == 'Fine-tuned LawBot + RAG'))
    lb_history = lb_history + [('User', message), ('LawBot', r['answer'])]
    shown = r['answer'] + '\n\n*Educational use only. Not legal advice.*'
    chat_history = chat_history + [{'role': 'user', 'content': message},
                                   {'role': 'assistant', 'content': shown}]
    return '', chat_history, lb_history, format_trace(r)

EXAMPLES = ['What is the punishment for murder?', 'What is anticipatory bail?',
            'Which article protects life and personal liberty?', 'What is the GST rate on gold jewellery?']

with gr.Blocks(title='LawBot') as demo:
    gr.Markdown('# ⚖️ LawBot: Indian Legal Q&A (IPC · CrPC · Constitution)\n'
                '**LawBot is for educational use only and is not legal advice.** Answers come from the Bare Act text '
                'and may be incomplete. The IPC and CrPC were replaced by the BNS and BNSS from 1 July 2024.')
    with gr.Row():
        with gr.Column(scale=3):
            try:
                chatbot = gr.Chatbot(type='messages', height=460, label='Conversation')
            except TypeError:                                   # newer Gradio: messages format is the default
                chatbot = gr.Chatbot(height=460, label='Conversation')
            msg = gr.Textbox(placeholder='Ask about the IPC, CrPC or the Constitution…', label='Your question')
            with gr.Row():
                send = gr.Button('Ask', variant='primary')
                clear = gr.Button('New conversation')
        with gr.Column(scale=2):
            model_choice = gr.Radio(['Base Qwen + RAG', 'Fine-tuned LawBot + RAG'],
                                    value='Base Qwen + RAG', label='Answer model')
            trace = gr.Markdown('Retrieved sources will appear here.')
            gr.Examples(EXAMPLES, inputs=msg)
    lb_state = gr.State([])

    send.click(respond, [msg, chatbot, lb_state, model_choice], [msg, chatbot, lb_state, trace])
    msg.submit(respond, [msg, chatbot, lb_state, model_choice], [msg, chatbot, lb_state, trace])
    clear.click(lambda: ([], [], 'Retrieved sources will appear here.'), None, [chatbot, lb_state, trace])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cee3eb9fbc5e220731.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
